# 42 — Connected Admissible Region Topology

**Engineering question**

How do we turn an admissible-region picture into reusable topology metrics?

Notebook 37 introduced operating regimes.  
Notebook 43 introduced robustness margins.  
Notebook 42 is the reusable topology notebook: it defines the computational tools that later notebooks use for region-constrained design search.

**Engineering statement**

> The largest connected admissible region is the engineering object that converts feasibility into robust operation.

## Setup

This notebook generates all figures and result files from code.

Outputs:

```text
figures/
results/csv/
results/json/
results/42_outputs.zip
```

In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import deque
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

def save_table(df, stem):
    csv_path = CSV / f"{stem}.csv"
    json_path = JS / f"{stem}.json"
    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient="records", indent=2)
    print(f"saved: {csv_path}")
    print(f"saved: {json_path}")

def grid(n=240):
    x = np.linspace(0, 1, n)
    y = np.linspace(0, 1, n)
    X, Y = np.meshgrid(x, y)
    return x, y, X, Y

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)

## 1. RegionDesign dataclass

Rather than passing loose dictionaries through the notebook, define a small reusable data structure.

The fields are intentionally generic: drive, loss, timing, calibration, routing, detection, coupling, cost, and control complexity.

In [ ]:
@dataclass
class RegionDesign:
    name: str
    drive_shift: float = 0.0
    loss_scale: float = 1.0
    timing_scale: float = 1.0
    calibration_error: float = 0.0
    routing_noise: float = 0.0
    detection_support: float = 1.0
    coupling_support: float = 1.0
    hardware_cost: float = 1.0
    control_complexity: float = 1.0

def admissibility_score(X, Y, design: RegionDesign):
    drive = np.clip(X + design.drive_shift, 0, 1.2)
    loss = np.clip(Y * design.loss_scale, 0, 1.4)

    enough_drive = 1 / (1 + np.exp(-18 * (drive - 0.30)))
    loss_survival = np.exp(-2.8 * loss)
    overdrive = np.exp(-9.0 * np.maximum(drive - 0.86, 0) ** 2)
    timing = np.exp(-design.timing_scale * 5.0 * np.maximum(drive + 0.74 * loss - 1.08, 0) ** 2)
    calibration = np.exp(-4.0 * design.calibration_error ** 2)
    broad_region = np.exp(-((drive - 0.62) ** 2 / 0.27 + (loss - 0.20) ** 2 / 0.18))
    routing_defects = 1 - design.routing_noise * np.sin(10 * X) * np.sin(8 * Y)

    score = (
        design.coupling_support
        * design.detection_support
        * enough_drive
        * loss_survival
        * overdrive
        * timing
        * calibration
        * (0.50 + 0.50 * broad_region)
        * routing_defects
    )
    score = np.clip(score, 0, None)
    return score / (score.max() + 1e-12)

designs = [
    RegionDesign("single resonator", drive_shift=-0.06, loss_scale=1.28, timing_scale=1.20, calibration_error=0.08, routing_noise=0.06, detection_support=0.92, coupling_support=0.94, hardware_cost=2, control_complexity=2),
    RegionDesign("coupled resonators", drive_shift=-0.02, loss_scale=1.10, timing_scale=1.05, calibration_error=0.05, routing_noise=0.04, detection_support=0.95, coupling_support=1.00, hardware_cost=4, control_complexity=4),
    RegionDesign("programmable mesh", drive_shift=0.02, loss_scale=1.00, timing_scale=0.90, calibration_error=0.04, routing_noise=0.02, detection_support=0.98, coupling_support=1.03, hardware_cost=7, control_complexity=7),
    RegionDesign("hybrid chip", drive_shift=0.04, loss_scale=0.82, timing_scale=0.72, calibration_error=0.02, routing_noise=0.01, detection_support=1.02, coupling_support=1.08, hardware_cost=9, control_complexity=8),
]

pd.DataFrame([asdict(d) for d in designs])

## 2. Connected-component analysis

The key reusable tool is `connected_components(mask)`.

It turns a binary admissibility mask into connected regions. The largest connected region is the object later notebooks call:

\[
\Omega_c
\]

or the largest connected admissible component.

In [ ]:
def connected_components(mask):
    visited = np.zeros_like(mask, dtype=bool)
    h, w = mask.shape
    comps = []

    for i in range(h):
        for j in range(w):
            if mask[i, j] and not visited[i, j]:
                q = deque([(i, j)])
                visited[i, j] = True
                pts = []
                while q:
                    a, b = q.popleft()
                    pts.append((a, b))
                    for da, db in [(1,0), (-1,0), (0,1), (0,-1)]:
                        na, nb = a + da, b + db
                        if 0 <= na < h and 0 <= nb < w and mask[na, nb] and not visited[na, nb]:
                            visited[na, nb] = True
                            q.append((na, nb))
                comps.append(pts)

    comps.sort(key=len, reverse=True)
    return comps

def largest_component_mask(mask):
    comps = connected_components(mask)
    out = np.zeros_like(mask, dtype=bool)
    if comps:
        pts = np.array(comps[0])
        out[pts[:,0], pts[:,1]] = True
    return out, comps

def boundary_distance(mask):
    h, w = mask.shape
    INF = h + w + 10
    dist = np.full((h, w), INF, dtype=float)
    q = deque()

    for i in range(h):
        for j in range(w):
            if not mask[i, j]:
                dist[i, j] = 0
                q.append((i, j))

    while q:
        i, j = q.popleft()
        for di, dj in [(1,0), (-1,0), (0,1), (0,-1)]:
            ni, nj = i + di, j + dj
            if 0 <= ni < h and 0 <= nj < w and dist[ni, nj] > dist[i, j] + 1:
                dist[ni, nj] = dist[i, j] + 1
                q.append((ni, nj))

    return dist

## 3. From admissibility field to topology

This figure shows the core transition:

```text
continuous admissibility field
        ↓
binary admitted region Ω
        ↓
largest connected component Ωc
```

In [ ]:
threshold = 0.50
x0, y0, X0, Y0 = grid(240)
demo = designs[-1]
Z0 = admissibility_score(X0, Y0, demo)
mask0 = Z0 >= threshold
largest0, comps0 = largest_component_mask(mask0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)

axes[0].imshow(Z0, origin="lower", extent=[0,1,0,1], aspect="auto", vmin=0, vmax=1)
axes[0].set_title("Admissibility field")
axes[0].set_ylabel("loss")

axes[1].imshow(mask0, origin="lower", extent=[0,1,0,1], aspect="auto", vmin=0, vmax=1)
axes[1].set_title("Admitted region Ω")

axes[2].imshow(largest0, origin="lower", extent=[0,1,0,1], aspect="auto", vmin=0, vmax=1)
axes[2].set_title("Largest connected region Ωc")

for ax in axes:
    ax.set_xlabel("drive")

fig.suptitle("From Admissibility to Connected Topology", fontsize=17, y=1.03)
savefig(fig, "42_01_admissibility_to_topology.png")
plt.show()

## 4. Region metrics

Separate topology metrics from geometry metrics.

Topology metrics:
- admitted area,
- largest connected component,
- component count.

Geometry metrics:
- average margin,
- maximum margin,
- maximum-margin point.

In [ ]:
def evaluate_region(design: RegionDesign, n=220, threshold=0.50):
    x, y, X, Y = grid(n)
    Z = admissibility_score(X, Y, design)
    mask = Z >= threshold
    largest, comps = largest_component_mask(mask)
    dist = boundary_distance(largest)

    if comps:
        pts = np.array(comps[0])
        largest_fraction = len(comps[0]) / mask.size
        admitted_area = mask.mean()
        avg_margin = dist[pts[:,0], pts[:,1]].mean() / max(mask.shape)
        max_margin = dist[pts[:,0], pts[:,1]].max() / max(mask.shape)
        best = pts[np.argmax(dist[pts[:,0], pts[:,1]])]
        x_star = x[best[1]]
        y_star = y[best[0]]
    else:
        admitted_area = largest_fraction = avg_margin = max_margin = x_star = y_star = 0.0

    return {
        "design": design.name,
        "admitted_area": admitted_area,
        "largest_component": largest_fraction,
        "component_count": len(comps),
        "average_margin": avg_margin,
        "maximum_margin": max_margin,
        "x_star": x_star,
        "y_star": y_star,
        "hardware_cost": design.hardware_cost,
        "control_complexity": design.control_complexity,
        "x": x, "y": y, "X": X, "Y": Y, "Z": Z, "mask": mask, "largest": largest, "dist": dist,
    }

evaluations = [evaluate_region(d) for d in designs]
summary_rows = [
    {k: v for k, v in ev.items() if k not in {"x","y","X","Y","Z","mask","largest","dist"}}
    for ev in evaluations
]
summary = pd.DataFrame(summary_rows)
save_table(summary, "42_connected_region_architecture_summary")
summary

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.6))
ax.axis("off")

display_df = summary.copy()
for c in ["admitted_area", "largest_component", "average_margin", "maximum_margin", "x_star", "y_star"]:
    display_df[c] = display_df[c].map(lambda v: f"{v:.3f}")

table = ax.table(
    cellText=display_df.values,
    colLabels=display_df.columns,
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(8.6)
table.scale(1.02, 1.60)

for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.0)
    if r == 0:
        cell.set_text_props(weight="bold")
    if c == 0 and r > 0:
        cell.set_text_props(ha="left")

ax.set_title("Architecture Summary from Connected-Region Metrics", fontsize=17, pad=18)
savefig(fig, "42_02_architecture_summary_table.png")
plt.show()

## 5. Topology versus geometry

Topology asks whether the admitted region is connected.

Geometry asks how far the chosen point is from failure boundaries.

Both are needed.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), sharex=True, sharey=True)

for ax, ev in zip(axes, evaluations):
    dist_norm = ev["dist"] / (ev["dist"].max() + 1e-12)
    dist_masked = np.where(ev["largest"], dist_norm, np.nan)
    ax.imshow(dist_masked, origin="lower", extent=[0,1,0,1], aspect="auto", vmin=0, vmax=1)
    ax.contour(ev["X"], ev["Y"], ev["Z"], levels=[threshold], colors="black", linewidths=1.7)
    ax.scatter([ev["x_star"]], [ev["y_star"]], s=100, color="red", zorder=4)
    ax.set_title(ev["design"])
    ax.set_xlabel("drive")
    if ax is axes[0]:
        ax.set_ylabel("loss")

fig.suptitle("Geometry Inside the Largest Connected Admissible Region", fontsize=17, y=1.03)
savefig(fig, "42_03_distance_fields_by_architecture.png")
plt.show()

## 6. Region fragmentation

This clean example distinguishes:

- one large connected region,
- a shrunk region,
- disconnected islands.

Robustness depends on the largest connected region, not merely admitted area.

In [ ]:
def gaussian_blob(X, Y, cx, cy, sx, sy, amp=1.0):
    return amp * np.exp(-((X-cx)**2/(2*sx**2) + (Y-cy)**2/(2*sy**2)))

def region_case(kind, n=220):
    x, y, X, Y = grid(n)
    if kind == "connected":
        Z = gaussian_blob(X,Y,0.47,0.28,0.18,0.16,1.0) + gaussian_blob(X,Y,0.62,0.31,0.18,0.15,0.95)
    elif kind == "shrunk":
        Z = gaussian_blob(X,Y,0.55,0.28,0.13,0.11,1.0)
    elif kind == "fragmented":
        Z = (
            gaussian_blob(X,Y,0.28,0.28,0.08,0.08,1.0)
            + gaussian_blob(X,Y,0.53,0.35,0.08,0.08,0.95)
            + gaussian_blob(X,Y,0.79,0.24,0.08,0.08,1.0)
        )
    else:
        raise ValueError(kind)
    return x, y, X, Y, Z / Z.max()

cases = ["connected", "shrunk", "fragmented"]
frag_rows = []
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), sharex=True, sharey=True)

for ax, case in zip(axes, cases):
    x, y, X, Y, Z = region_case(case)
    mask = Z >= 0.50
    comps = connected_components(mask)
    largest = len(comps[0]) / mask.size if comps else 0
    area = mask.mean()
    frag_rows.append({"case": case, "admitted_area": area, "largest_component": largest, "component_count": len(comps)})
    ax.imshow(Z, origin="lower", extent=[0,1,0,1], aspect="auto", vmin=0, vmax=1)
    ax.contour(X, Y, Z, levels=[0.50], colors="black", linewidths=2.2)
    ax.set_title(case)
    ax.set_xlabel("drive")
    if ax is axes[0]:
        ax.set_ylabel("loss")
    ax.text(0.05, 0.88, f"components: {len(comps)}", transform=ax.transAxes, fontsize=10)
    ax.text(0.05, 0.78, f"largest: {largest:.2f}", transform=ax.transAxes, fontsize=10)

fig.suptitle("Fragmentation Reduces Connected Admissible Topology", fontsize=17, y=1.03)
savefig(fig, "42_04_region_fragmentation.png")
plt.show()

fragmentation = pd.DataFrame(frag_rows)
save_table(fragmentation, "42_region_fragmentation")
fragmentation

## 7. Sensitivity analysis

Sensitivity asks which engineering action expands the largest connected component.

This turns qualitative priorities into a computable ranking.

In [ ]:
def modified_design(base: RegionDesign, action):
    d = RegionDesign(**asdict(base))
    if action == "loss reduction":
        d.loss_scale *= 0.86
    elif action == "timing control":
        d.timing_scale *= 0.75
    elif action == "calibration stabilization":
        d.calibration_error *= 0.25
    elif action == "routing improvement":
        d.routing_noise *= 0.25
    elif action == "detection improvement":
        d.detection_support *= 1.05
    elif action == "coupling improvement":
        d.coupling_support *= 1.05
    else:
        raise ValueError(action)
    return d

base = designs[1]  # coupled resonators
actions = ["loss reduction", "timing control", "calibration stabilization", "routing improvement", "detection improvement", "coupling improvement"]

base_eval = evaluate_region(base)
sens_rows = []
for action in actions:
    improved = modified_design(base, action)
    ev = evaluate_region(improved)
    sens_rows.append({
        "action": action,
        "delta_largest_component": ev["largest_component"] - base_eval["largest_component"],
        "delta_maximum_margin": ev["maximum_margin"] - base_eval["maximum_margin"],
        "baseline_largest_component": base_eval["largest_component"],
        "improved_largest_component": ev["largest_component"],
    })

sensitivity = pd.DataFrame(sens_rows).sort_values("delta_largest_component", ascending=False)
save_table(sensitivity, "42_connected_region_sensitivity")
sensitivity

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.5))
plot_df = sensitivity.iloc[::-1]
ax.barh(plot_df["action"], plot_df["delta_largest_component"])
ax.set_xlabel("increase in largest connected admissible region")
ax.set_title("Sensitivity of Connected Region to Engineering Action", fontsize=17)
ax.grid(axis="x", alpha=0.25)

xmax = max(plot_df["delta_largest_component"].max(), 0.001)
ax.set_xlim(0, xmax * 1.18)
for action, value in zip(plot_df["action"], plot_df["delta_largest_component"]):
    ax.text(value + xmax * 0.015, action, f"{value:.3f}", va="center", fontsize=10)

savefig(fig, "42_05_connected_region_sensitivity.png")
plt.show()

## 8. Topology metric ladder

The notebook's reusable result is a metric ladder:

```text
admissibility field
→ admitted mask Ω
→ connected components
→ largest component Ωc
→ distance-to-boundary
→ maximum-margin point x*
```

In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 4.8))
ax.axis("off")

ladder = [
    ("Admissibility\nfield", "continuous score"),
    ("Ω", "admitted mask"),
    ("Components", "topology"),
    ("Ωc", "largest connected region"),
    ("d(∂Ωc)", "distance field"),
    ("x*", "maximum-margin point"),
]
xs = np.linspace(0.08, 0.92, len(ladder))
y = 0.55
w, h = 0.13, 0.28

for i, (title, subtitle) in enumerate(ladder):
    ax.add_patch(Rectangle((xs[i]-w/2, y-h/2), w, h, fill=False, linewidth=2.2))
    ax.text(xs[i], y+0.045, title, ha="center", va="center", fontsize=10.5, weight="bold")
    ax.text(xs[i], y-0.07, subtitle, ha="center", va="center", fontsize=8.6)
    if i < len(ladder)-1:
        ax.annotate("", xy=(xs[i+1]-w/2-0.008, y), xytext=(xs[i]+w/2+0.008, y),
                    arrowprops=dict(arrowstyle="->", linewidth=2.0))

ax.text(0.5, 0.16, "Topology specifies the robust operating point.", ha="center", fontsize=12)
ax.set_title("Connected-Region Topology Metric Ladder", fontsize=18)
savefig(fig, "42_06_topology_metric_ladder.png")
plt.show()

## 9. Export bundle

In [ ]:
review = {
    "architecture_summary": summary.to_dict(orient="records"),
    "fragmentation": fragmentation.to_dict(orient="records"),
    "sensitivity": sensitivity.to_dict(orient="records"),
}

with open(JS / "42_review_bundle.json", "w", encoding="utf-8") as f:
    json.dump(review, f, indent=2)

zip_path = RES / "42_outputs.zip"
files_to_zip = list(FIG.glob("42_*.png")) + list(CSV.glob("42_*.csv")) + list(JS.glob("42_*.json"))

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

## Takeaways

- The largest connected admissible component \(\Omega_c\) is the central engineering object.
- Topology metrics and geometry metrics should be separated.
- Topology answers: is the admitted region connected?
- Geometry answers: how far is the operating point from failure?
- Sensitivity analysis ranks engineering actions by how much they expand \(\Omega_c\).

This notebook is deliberately reusable: later notebooks can import the same conceptual functions for search, robustness, and architecture comparison.